# UdaPlay 02 Solution Project: Agent Implementation

This notebook demonstrates a stateful research agent with internal retrieval, retrieval evaluation, Tavily web fallback, persistent memory, and structured cited answers.

In [1]:
from dotenv import load_dotenv
load_dotenv('.env')
import json, sys
from pathlib import Path
sys.path.insert(0, str(Path('src').resolve()))

In [2]:
from udaplay import UdaPlayAgent, VectorStoreManager

manager = VectorStoreManager(data_dir='data/games', chroma_dir='chroma_db')
if manager.collection.count() == 0:
    manager.build_index(reset=True)
agent = UdaPlayAgent(vector_store=manager, memory_path='memory/learned_facts.jsonl')

## Tool Demonstration

The agent exposes the required tools: `retrieve_game`, `evaluate_retrieval`, and `game_web_search`.

In [3]:
retrieved = agent.retrieve_game('When was God of War Ragnarok released?')
print(json.dumps(agent.evaluate_retrieval('When was God of War Ragnarok released?', retrieved), indent=2))

{
  "confidence": 0.647,
  "needs_web": false,
  "reason": "Best internal score=0.591, lexical_overlap=0.750.",
  "matched_terms": [
    "ragnarok",
    "war",
    "god"
  ]
}


## Example Queries and Reports

In [4]:
queries = [
    'Who developed FIFA 21?',
    'When was God of War Ragnarok released?',
    'What platform was Pokémon Red launched on?',
    'What is Rockstar Games working on right now?'
]
for query in queries:
    report = agent.answer(query)
    print('\n' + '=' * 80)
    print(json.dumps(report.model_dump(), indent=2, ensure_ascii=False))


{
  "question": "Who developed FIFA 21?",
  "answer": "FIFA 21 was developed by EA Vancouver and EA Romania.",
  "confidence": 0.552,
  "used_web_search": false,
  "citations": [
    {
      "title": "FIFA 21",
      "source": "Internal UdaPlay curated dataset",
      "source_type": "internal"
    }
  ],
  "reasoning": [
    "START -> RETRIEVE: Searching the internal UdaPlay knowledge base.",
    "RETRIEVE -> EVALUATE: Best internal score=0.491, lexical_overlap=0.667.",
    "EVALUATE -> REPORT: Internal knowledge was sufficient."
  ],
  "tool_usage": [
    "retrieve_game",
    "evaluate_retrieval"
  ],
  "structured_facts": {
    "title": "FIFA 21",
    "developer": "EA Vancouver and EA Romania",
    "publisher": "EA Sports",
    "release_date": "2020-10-09",
    "platforms": "PlayStation 4, Xbox One, Nintendo Switch, Windows, PlayStation 5, Xbox Series X/S, Stadia",
    "genres": "Sports, Football simulation"
  }
}

{
  "question": "When was God of War Ragnarok released?",
  "answer"

## Conversation State

The agent remembers prior turns and stores learned web facts in long-term memory.

In [5]:
agent.state.conversation[-4:]


[{'role': 'user', 'content': 'What platform was Pokémon Red launched on?'},
 {'role': 'assistant',
  'content': 'Pokemon Red launched on or is available for: Game Boy.'},
 {'role': 'user', 'content': 'What is Rockstar Games working on right now?'},
 {'role': 'assistant',
  'content': 'Publicly discussed Rockstar Games work centers on Grand Theft Auto VI, with a planned release window announced for current-generation consoles. Internal context also matched Grand Theft Auto VI.'}]